# What drives housing prices across European countries?

## Introduction

This project analyzes the relationship between GDP per capita and housing prices across European countries using Eurostat data.

## Research Question

How is GDP per capita related to housing prices, and how does this relationship vary across time and country groups?

In [4]:
!pip install pandas
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 5.7 MB/s  0:00:01a 0:00:01m eta 0:00:01


In [7]:
housing = pd.read_csv("../data/housing.csv")
gdp = pd.read_csv("../data/gdp_raw.csv")

housing.head()

,country,country_type,eu_member,eurozone_member,year,quarter_num,quarter,price_index,quarterly_change_pct,yearly_change_pct,price_change_since_2015_pct,data_quality
0,Austria,Individual,Yes,Yes,2022,4,2022-Q4,166.32,-4.2,5.7,66.32,Complete
1,Austria,Individual,Yes,Yes,2023,1,2023-Q1,164.35,-1.2,-0.2,64.35,Complete
2,Austria,Individual,Yes,Yes,2023,2,2023-Q2,164.68,0.2,-2.8,64.68,Complete
3,Austria,Individual,Yes,Yes,2023,3,2023-Q3,164.29,-0.2,-5.4,64.29,Complete
4,Austria,Individual,Yes,Yes,2023,4,2023-Q4,161.40,-1.8,-3.0,61.40,Complete


In [8]:
print("HOUSING SHAPE:", housing.shape)
print("GDP SHAPE:", gdp.shape)

print("\nHOUSING COLUMNS:")
print(housing.columns)

print("\nGDP COLUMNS:")
print(gdp.columns)

HOUSING SHAPE: (417, 12)
GDP SHAPE: (425, 19)

HOUSING COLUMNS:
Index(['country', 'country_type', 'eu_member', 'eurozone_member', 'year',
       'quarter_num', 'quarter', 'price_index', 'quarterly_change_pct',
       'yearly_change_pct', 'price_change_since_2015_pct', 'data_quality'],
      dtype='str')

GDP COLUMNS:
Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency',
       'unit', 'Unit of measure', 'na_item',
       'National accounts indicator (ESA 2010)', 'geo',
       'Geopolitical entity (reporting)', 'TIME_PERIOD', 'Time', 'OBS_VALUE',
       'Observation value', 'OBS_FLAG',
       'Observation status (Flag) V2 structure', 'CONF_STATUS',
       'Confidentiality status (flag)'],
      dtype='str')


In [9]:
gdp_clean = gdp[["geo", "TIME_PERIOD", "OBS_VALUE"]].copy()

gdp_clean.columns = ["country_code", "year", "gdp_per_capita"]

gdp_clean["year"] = gdp_clean["year"].astype(int)
gdp_clean["gdp_per_capita"] = pd.to_numeric(gdp_clean["gdp_per_capita"], errors="coerce")

gdp_clean = gdp_clean.dropna()

gdp_clean.head()

,country_code,year,gdp_per_capita
0,AL,2016,517470
1,AL,2017,549530
2,AL,2018,579410
3,AL,2019,599830
4,AL,2020,583540


In [10]:
country_map = {
    "AT": "Austria",
    "DE": "Germany",
    "FR": "France",
    "IT": "Italy",
    "ES": "Spain",
    "BG": "Bulgaria",
    "RO": "Romania",
    "PL": "Poland",
    "NL": "Netherlands",
    "BE": "Belgium",
    "SE": "Sweden",
    "FI": "Finland",
    "DK": "Denmark",
    "GR": "Greece",
    "PT": "Portugal",
    "IE": "Ireland"
}

gdp_clean["country"] = gdp_clean["country_code"].map(country_map)

gdp_clean = gdp_clean.dropna(subset=["country"])

gdp_clean.head()

,country_code,year,gdp_per_capita,country
7,AT,2016,40690,Austria
8,AT,2017,41760,Austria
9,AT,2018,43360,Austria
10,AT,2019,44570,Austria
11,AT,2020,42650,Austria


In [11]:
housing_yearly = housing.groupby(
    ["country", "year"],
    as_index=False
)["price_index"].mean()

housing_yearly.head()

,country,year,price_index
0,Austria,2022,166.320000
1,Austria,2023,163.680000
2,Austria,2024,163.060000
3,Austria,2025,167.253333
4,Belgium,2022,134.550000


In [14]:
df = housing_yearly.merge(
    gdp_clean,
    on=["country", "year"],
    how="inner"
)

df.head()

,country,year,price_index,country_code,gdp_per_capita
0,Austria,2022,166.320000,AT,49640
1,Austria,2023,163.680000,AT,52330
2,Austria,2024,163.060000,AT,53830
3,Austria,2025,167.253333,AT,55710
4,Belgium,2022,134.550000,BE,48090


In [15]:
print("HOUSING shape:", housing.shape)
print("GDP shape:", gdp_clean.shape)
print("MERGED shape:", df.shape)

print("\nHOUSING preview:")
display(housing.head())

print("\nGDP preview:")
display(gdp_clean.head())

print("\nMERGED preview:")
display(df.head())

print("\nUnique countries in merged:")
print(df["country"].unique())

print("\nNumber of countries in merged:", df["country"].nunique())

HOUSING shape: (417, 12)
GDP shape: (150, 4)
MERGED shape: (60, 5)

HOUSING preview:


,country,country_type,eu_member,eurozone_member,year,quarter_num,quarter,price_index,quarterly_change_pct,yearly_change_pct,price_change_since_2015_pct,data_quality
0,Austria,Individual,Yes,Yes,2022,4,2022-Q4,166.32,-4.2,5.7,66.32,Complete
1,Austria,Individual,Yes,Yes,2023,1,2023-Q1,164.35,-1.2,-0.2,64.35,Complete
2,Austria,Individual,Yes,Yes,2023,2,2023-Q2,164.68,0.2,-2.8,64.68,Complete
3,Austria,Individual,Yes,Yes,2023,3,2023-Q3,164.29,-0.2,-5.4,64.29,Complete
4,Austria,Individual,Yes,Yes,2023,4,2023-Q4,161.40,-1.8,-3.0,61.40,Complete



GDP preview:


,country_code,year,gdp_per_capita,country
7,AT,2016,40690,Austria
8,AT,2017,41760,Austria
9,AT,2018,43360,Austria
10,AT,2019,44570,Austria
11,AT,2020,42650,Austria



MERGED preview:


,country,year,price_index,country_code,gdp_per_capita
0,Austria,2022,166.320000,AT,49640
1,Austria,2023,163.680000,AT,52330
2,Austria,2024,163.060000,AT,53830
3,Austria,2025,167.253333,AT,55710
4,Belgium,2022,134.550000,BE,48090



Unique countries in merged:
<StringArray>
[    'Austria',     'Belgium',    'Bulgaria',     'Denmark',     'Finland',
      'France',     'Germany',     'Ireland',       'Italy', 'Netherlands',
      'Poland',    'Portugal',     'Romania',       'Spain',      'Sweden']
Length: 15, dtype: str

Number of countries in merged: 15
